In [69]:
import pandas as pd 
from clean_weather import extract_current_weather_data
from clean_weather import load_raw_files
import os

In [57]:
path=r"F:\pakistan_weather_pipeline\data\raw"
df=extract_current_weather_data(load_raw_files(path))

In [58]:
def extract_hourly_weather(all_data):
    hourly_frames = []
    for data in all_data:
        hourly_df=pd.DataFrame(data['hourly'])
        hourly_df['city']=data['city']
        hourly_frames.append(hourly_df)
    return pd.concat(hourly_frames, ignore_index=True)


In [59]:
all_data = load_raw_files(path)
hourly_df = extract_hourly_weather(all_data)
print(hourly_df.shape)   
hourly_df

(840, 6)


,time,temperature_2m,precipitation,windspeed_10m,weathercode,city
0,2026-07-20T00:00,26.7,1.1,13.5,55,Islamabad
1,2026-07-20T01:00,26.6,0.3,12.3,51,Islamabad
2,2026-07-20T02:00,25.2,5.2,9.9,96,Islamabad
3,2026-07-20T03:00,25.3,1.1,10.8,55,Islamabad
4,2026-07-20T04:00,24.9,0.4,8.8,51,Islamabad
...,...,...,...,...,...,...
835,2026-07-26T19:00,32.5,0.0,6.7,0,Quetta
836,2026-07-26T20:00,30.4,0.0,7.3,0,Quetta
837,2026-07-26T21:00,28.3,0.0,8.9,0,Quetta
838,2026-07-26T22:00,26.4,0.0,10.8,0,Quetta


In [65]:
def clean_current_weather(df):
    WEATHER_LABELS = {
        0: "Clear", 1: "Mainly Clear", 2: "Partly Cloudy",
        3: "Overcast", 51: "Drizzle", 53: "Moderate Drizzle",
        55: "Dense Drizzle", 80: "Rain Showers", 95: "Thunderstorm",
        96: "Thunderstorm with Hail"
    }
    
    # 1. Create an independent clone to protect your original raw data
    cleaned_df = df.copy()
    
    # 2. Modify the copy (inplace=True is now safe)
    cleaned_df.rename(columns={'time': 'recorded_at'}, inplace=True)

    cleaned_df['load_timestamp']=pd.Timestamp.now()
    
    # Fixed: Wrapped 'interval' in a list just to follow standard Pandas practice
    cleaned_df.drop(columns=['interval'], inplace=True)
    
    # 3. Map the weather codes to human-readable strings
    cleaned_df['weather_description'] = cleaned_df['weathercode'].map(WEATHER_LABELS)
    
    return cleaned_df

cleaned_df=clean_current_weather(df)

In [66]:
def clean_hourly_weather(hourly_df):
    WEATHER_LABELS = {
        0: "Clear", 1: "Mainly Clear", 2: "Partly Cloudy",
        3: "Overcast", 51: "Drizzle", 53: "Moderate Drizzle",
        55: "Dense Drizzle", 80: "Rain Showers", 95: "Thunderstorm",
        96: "Thunderstorm with Hail"
    }    
    cleaned_hourly_df=hourly_df.copy()
    cleaned_hourly_df.rename(columns={'time':'forcast_time','temperature_2m':'temperature','windspeed_10m':'windspeed'},inplace=True)
    cleaned_hourly_df['load_timestamp']=pd.Timestamp.now()
    cleaned_hourly_df['weather_discription']=cleaned_hourly_df['weathercode'].map(WEATHER_LABELS)
    return cleaned_hourly_df
cleaned_hourly_df=clean_hourly_weather(hourly_df)    

In [70]:
def save_clean_data(cleaned_df, cleaned_hourly_df):
    """
    Saves the cleaned current and hourly DataFrames to CSV files.
    Output location: data/clean/
    index=False prevents pandas adding an extra numbered column
    """
    # Create output folder if it does not exist
    os.makedirs(r"F:\pakistan_weather_pipeline\data\clean", exist_ok=True)

    cleaned_df.to_csv(
        r"F:\pakistan_weather_pipeline\data\clean\current_weather.csv",
        index=False
    )

    cleaned_hourly_df.to_csv(
        r"F:\pakistan_weather_pipeline\data\clean\hourly_weather.csv",
        index=False
    )

In [71]:
save_clean_data(cleaned_df,cleaned_hourly_df)